<div dir="rtl" align="right">

# ریاضی طبقه‌بندی: سیگموید، softmax و گرادیان

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## از نمره به احتمال
طبقه‌بند خطی اول یک **نمره** می‌سازد: $z=w\cdot x+b$. این نمره هر عددی می‌تواند باشد؛ برای تبدیل آن به احتمال از سیگموید استفاده می‌کنیم:

$$p=\sigma(z)=\frac{1}{1+e^{-z}}$$

معنی z: **لگاریتم بخت** (log-odds). اگر $p=0.8$ باشد، بخت $0.8/0.2=4$ و $z=\ln4=1.386$.

| z | −3 | −1 | 0 | 1 | 3 |
|---|---|---|---|---|---|
| p | 0.047 | 0.269 | 0.5 | 0.731 | 0.953 |
| بخت $p/(1-p)$ | 0.05 | 0.37 | 1 | 2.72 | 20.1 |

## تابع خطا
از درس درست‌نمایی: آنتروپی متقاطع دودویی

$$L=-\big[y\ln p+(1-y)\ln(1-p)\big]$$

> [خطای لگاریتمی: اگر مدل با اطمینان اشتباه کند، جریمه بسیار بزرگ می‌شود.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/logit_loss.png)

*خطای لگاریتمی: اگر مدل با اطمینان اشتباه کند، جریمه بسیار بزرگ می‌شود.*

## گرادیان: یک نتیجه زیبا
با قاعده زنجیره‌ای $\dfrac{\partial L}{\partial z}=\dfrac{\partial L}{\partial p}\cdot\dfrac{\partial p}{\partial z}$:

| جزء | مقدار |
|---|---|
| $\partial L/\partial p$ | $-\dfrac{y}{p}+\dfrac{1-y}{1-p}=\dfrac{p-y}{p(1-p)}$ |
| $\partial p/\partial z$ | $p(1-p)$ |
| حاصل‌ضرب | $p-y$ |

$$\frac{\partial L}{\partial z}=p-y\qquad\frac{\partial L}{\partial w}=(p-y)\,x\qquad\frac{\partial L}{\partial b}=p-y$$

گرادیان فقط «پیش‌بینی منهای واقعیت» ضرب در ورودی است؛ دقیقاً همان شکلی که در رگرسیون خطی داشتیم.

## مثال عددی: یک گام آموزش
$x=(2,1)$، $y=1$، $w=(0.5,-1)$، $b=0.5$، $\eta=0.1$:

| گام | محاسبه | مقدار |
|---|---|---|
| نمره z | $0.5\times2-1\times1+0.5$ | 0.5 |
| احتمال p | $\sigma(0.5)$ | 0.622 |
| خطا | $-\ln0.622$ | 0.474 |
| $p-y$ | $0.622-1$ | −0.378 |
| $\partial L/\partial w$ | $-0.378\times(2,1)$ | $(-0.755,\ -0.378)$ |
| w جدید | $(0.5,-1)-0.1(-0.755,-0.378)$ | $(0.576,\ -0.962)$ |
| b جدید | $0.5+0.0378$ | 0.538 |
| p جدید | $\sigma(0.727)$ | 0.674 |
| خطای جدید | $-\ln0.674$ | 0.394 |

خطا از 0.474 به 0.394 رسید.

## بیش از دو کلاس: softmax
برای K کلاس، K نمره داریم و softmax آن‌ها را به توزیع احتمال تبدیل می‌کند:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}$$

> آزمایشگاه تعاملی این بخش را در سایت باز کن: https://behrooztaheri.github.io/learn-python-ai/#/l/math-logit

مثال با $z=(2,\ 1,\ 0.1)$:

| کلاس | $z_k$ | $e^{z_k}$ | $p_k$ |
|---|---|---|---|
| 1 | 2 | 7.389 | 0.659 |
| 2 | 1 | 2.718 | 0.242 |
| 3 | 0.1 | 1.105 | 0.099 |
| جمع | | 11.212 | 1 |

اگر کلاس درست کلاس 1 باشد: خطا $=-\ln0.659=0.417$ و گرادیان باز هم همان شکل ساده را دارد:

$$\frac{\partial L}{\partial z}=p-y=(0.659-1,\ 0.242,\ 0.099)=(-0.341,\ 0.242,\ 0.099)$$

## ویژگی‌های softmax
| ویژگی | توضیح |
|---|---|
| جمع = 1 | یک توزیع احتمال معتبر |
| حفظ ترتیب | بزرگ‌ترین نمره، بزرگ‌ترین احتمال |
| حساس به اختلاف | اضافه کردن یک عدد ثابت به همه نمره‌ها چیزی را عوض نمی‌کند |
| دما T | $z/T$: دمای کم = قاطع‌تر، دمای زیاد = یکنواخت‌تر (در مدل‌های زبانی، «خلاقیت») |
| K = 2 | همان سیگموید است |

> در کد هیچ‌وقت $e^{z}$ را مستقیم حساب نکن: برای $z=1000$ سرریز می‌شود. اول بزرگ‌ترین نمره را کم کن (`z - z.max()`). کتابخانه‌ها softmax و لگاریتم را با هم و به‌صورت پایدار حساب می‌کنند (log-sum-exp).

## مرز تصمیم
مدل وقتی کلاس 1 را انتخاب می‌کند که $p>0.5$ یعنی $z>0$ یعنی $w\cdot x+b>0$. پس مرز تصمیم **یک خط** (یا ابرصفحه) است که بر w عمود است؛ به همین دلیل رگرسیون لجستیک یک طبقه‌بند **خطی** است.



</div>

In [ ]:
import numpy as np

sigmoid = lambda z: 1 / (1 + np.exp(-z))

# One training step of logistic regression, as in the lesson
x = np.array([2.0, 1.0]); y = 1.0
w = np.array([0.5, -1.0]); b = 0.5; eta = 0.1
for step in range(3):
    z = w @ x + b; p = sigmoid(z)
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    print(f"step {step}: z = {z:.3f}  p = {p:.3f}  loss = {loss:.3f}  grad_w = {((p - y) * x).round(3)}")
    w = w - eta * (p - y) * x
    b = b - eta * (p - y)

# Check that dL/dz = p - y with a numerical derivative
L = lambda z: -np.log(sigmoid(z))               # y = 1
h = 1e-6
print("numeric dL/dz at z = 0.5:", round((L(0.5 + h) - L(0.5 - h)) / (2 * h), 4), "   p - y =", round(sigmoid(0.5) - 1, 4))

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=float) / T
    e = np.exp(z - z.max())                      # subtract the max: numerically stable
    return e / e.sum()

z = np.array([2.0, 1.0, 0.1])
p = softmax(z)
print("softmax:", p.round(3), "  sum =", p.sum())
print("loss if class 1 is true:", round(-np.log(p[0]), 3), "  gradient p - y:", (p - np.array([1, 0, 0])).round(3))
for T in [0.2, 1, 5]:
    print(f"T = {T}:", softmax(z, T).round(3))
print("huge scores still work:", softmax([1000, 999, 0]).round(3))

# Train logistic regression from scratch on two clouds of points
rng = np.random.default_rng(0)
X = np.r_[rng.normal([-1, -1], 1, (100, 2)), rng.normal([1.5, 1], 1, (100, 2))]
t = np.r_[np.zeros(100), np.ones(100)]
w = np.zeros(2); b = 0.0
for epoch in range(300):
    p = sigmoid(X @ w + b)
    w -= 0.1 * X.T @ (p - t) / 200
    b -= 0.1 * np.mean(p - t)
print("w =", w.round(3), " b =", round(b, 3), " accuracy =", np.mean((sigmoid(X @ w + b) > 0.5) == t))

<div dir="rtl" align="right">

### تمرین

برای $z=(1,2,3)$ خروجی softmax را با دست حساب کن. اگر کلاس درست کلاس 3 باشد خطا چقدر است؟

</div>

In [ ]:
# Your answer here

<div dir="rtl" align="right">

### تمرین

در آزمایشگاه دما را به 0.1 و سپس به 5 ببر. چه تفاوتی می‌بینی؟

</div>

In [ ]:
# Your answer here